In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import pickle
import os

In [2]:
df = pd.read_csv('../data/PS2_Dataset.csv')
print(df.shape)
print(df.head())

(6901, 20)
   Logical quotient rating  hackathons  coding skills rating  \
0                        5           0                     6   
1                        7           6                     4   
2                        2           3                     9   
3                        2           6                     3   
4                        2           0                     3   

   public speaking points self-learning capability? Extra-courses did  \
0                       2                       yes                no   
1                       3                        no               yes   
2                       1                        no               yes   
3                       5                        no               yes   
4                       4                       yes                no   

         certifications          workshops reading and writing skills  \
0  information security            testing                       poor   
1     shell program

In [3]:
cat_cols = df.select_dtypes(include='object').columns.tolist()

for col in cat_cols:
    print(f"{col}: {df[col].unique()}")
    print()

self-learning capability?: <ArrowStringArray>
['yes', 'no']
Length: 2, dtype: str

Extra-courses did: <ArrowStringArray>
['no', 'yes']
Length: 2, dtype: str

certifications: <ArrowStringArray>
['information security',    'shell programming',        'r programming',
        'distro making',     'machine learning',           'full stack',
               'hadoop',      'app development',               'python']
Length: 9, dtype: str

workshops: <ArrowStringArray>
[          'testing', 'database security',  'game development',
      'data science',  'system designing',           'hacking',
   'cloud computing',  'web technologies']
Length: 8, dtype: str

reading and writing skills: <ArrowStringArray>
['poor', 'excellent', 'medium']
Length: 3, dtype: str

memory capability score: <ArrowStringArray>
['poor', 'medium', 'excellent']
Length: 3, dtype: str

Interested subjects: <ArrowStringArray>
[          'programming',            'Management',      'data engineering',
              'networks'

C:\Users\Siddhi\AppData\Local\Temp\ipykernel_31456\2806198892.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns.tolist()


In [4]:
df_encoded = df.copy()
encoders = {}

# Manual encoding for ordinal columns (order matters)
ordinal_mapping = {
    'reading and writing skills': {'poor': 0, 'medium': 1, 'excellent': 2},
    'memory capability score':    {'poor': 0, 'medium': 1, 'excellent': 2},
}

for col, mapping in ordinal_mapping.items():
    df_encoded[col] = df_encoded[col].map(mapping)
    encoders[col] = mapping  # save mapping dict instead of LabelEncoder
    print(f"Ordinal encoded: {col}")

# Regular LabelEncoder for all other categorical columns
cat_cols = df.select_dtypes(include='object').columns.tolist()
remaining_cats = [c for c in cat_cols if c not in ordinal_mapping]

for col in remaining_cats:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    encoders[col] = le
    print(f"Label encoded: {col}")

print("\nAll encoding done!")
df_encoded.head()

Ordinal encoded: reading and writing skills
Ordinal encoded: memory capability score
Label encoded: self-learning capability?
Label encoded: Extra-courses did
Label encoded: certifications
Label encoded: workshops
Label encoded: Interested subjects
Label encoded: interested career area 
Label encoded: Type of company want to settle in?
Label encoded: Taken inputs from seniors or elders


C:\Users\Siddhi\AppData\Local\Temp\ipykernel_31456\1650326436.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns.tolist()


Label encoded: Interested Type of Books
Label encoded: Management or Technical
Label encoded: hard/smart worker
Label encoded: worked in teams ever?
Label encoded: Introvert
Label encoded: Suggested Job Role

All encoding done!


,Logical quotient rating,hackathons,coding skills rating,public speaking points,self-learning capability?,Extra-courses did,certifications,workshops,reading and writing skills,memory capability score,Interested subjects,interested career area,Type of company want to settle in?,Taken inputs from seniors or elders,Interested Type of Books,Management or Technical,hard/smart worker,worked in teams ever?,Introvert,Suggested Job Role
0,5,0,6,2,1,0,4,6,0,0,9,5,0,0,28,0,1,1,0,0
1,7,6,4,3,0,1,8,6,2,1,2,4,1,1,3,1,0,0,1,0
2,2,3,9,1,0,1,4,6,2,0,5,0,9,1,29,1,1,0,0,0
3,2,6,3,5,0,1,7,2,2,0,7,5,7,1,13,0,1,1,1,0
4,2,0,3,4,1,0,1,3,2,1,3,4,0,0,14,1,0,1,0,0


In [5]:
# Should show 0, 1, 2 for poor, medium, excellent
print(df_encoded['reading and writing skills'].unique())
print(df_encoded['memory capability score'].unique())

# Check no nulls were introduced
print("\nAny nulls after encoding?")
print(df_encoded.isnull().sum().sum())  # should print 0

[0 2 1]
[0 1 2]

Any nulls after encoding?
0


In [6]:
X = df_encoded.drop('Suggested Job Role', axis=1)
y = df_encoded['Suggested Job Role']

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget classes:", encoders['Suggested Job Role'].classes_)

Features shape: (6901, 19)
Target shape: (6901,)

Target classes: ['Applications Developer' 'CRM Technical Developer' 'Database Developer'
 'Mobile Applications Developer' 'Network Security Engineer'
 'Software Developer' 'Software Engineer'
 'Software Quality Assurance (QA) / Testing'
 'Systems Security Administrator' 'Technical Support' 'UX Designer'
 'Web Developer']


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # 80% train, 20% test
    random_state=42,    # for reproducibility
    stratify=y          # keeps class balance in both splits
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (5520, 19)
X_test: (1381, 19)
y_train: (5520,)
y_test: (1381,)


In [8]:
# Create models folder if not exists
os.makedirs('../models', exist_ok=True)

# Save encoders — we'll need these in the Streamlit app
with open('../models/encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)

# Save processed data splits
X_train.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

print("Everything saved successfully!")

Everything saved successfully!


In [9]:
# Add this cell and run it
import pickle
import pandas as pd

with open('../models/encoders.pkl', 'rb') as f:
    encoders = pickle.load(f)

y_train = pd.read_csv('../data/y_train.csv').squeeze()

print("y_train sample:", y_train.head(10).tolist())
print("y_train dtype:", y_train.dtype)
print("encoder classes:", encoders['Suggested Job Role'].classes_)

y_train sample: [6, 4, 6, 7, 1, 0, 0, 4, 4, 8]
y_train dtype: int64
encoder classes: ['Applications Developer' 'CRM Technical Developer' 'Database Developer'
 'Mobile Applications Developer' 'Network Security Engineer'
 'Software Developer' 'Software Engineer'
 'Software Quality Assurance (QA) / Testing'
 'Systems Security Administrator' 'Technical Support' 'UX Designer'
 'Web Developer']


In [10]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X_train = pd.read_csv('../data/X_train.csv')
X_test  = pd.read_csv('../data/X_test.csv')
y_train = pd.read_csv('../data/y_train.csv').squeeze()
y_test  = pd.read_csv('../data/y_test.csv').squeeze()

# Check everything
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("y_train unique:", sorted(y_train.unique().tolist()))
print("y_test unique:", sorted(y_test.unique().tolist()))

# Check if any NaN in X_train
print("\nNaN in X_train:", X_train.isnull().sum().sum())
print("NaN in X_test:", X_test.isnull().sum().sum())

# Check X_train column names
print("\nX_train columns:", X_train.columns.tolist())

# Quick manual test
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
pred = dt.predict(X_test)

print("\ny_test first 10:", y_test.head(10).tolist())
print("pred first 10:", pred[:10].tolist())
print("\nAccuracy:", accuracy_score(y_test, pred))

X_train shape: (5520, 19)
y_train shape: (5520,)
y_train unique: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
y_test unique: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]

NaN in X_train: 0
NaN in X_test: 0

X_train columns: ['Logical quotient rating', 'hackathons', 'coding skills rating', 'public speaking points', 'self-learning capability?', 'Extra-courses did', 'certifications', 'workshops', 'reading and writing skills', 'memory capability score', 'Interested subjects', 'interested career area ', 'Type of company want to settle in?', 'Taken inputs from seniors or elders', 'Interested Type of Books', 'Management or Technical', 'hard/smart worker', 'worked in teams ever?', 'Introvert']

y_test first 10: [2, 10, 1, 11, 5, 8, 8, 6, 7, 11]
pred first 10: [0, 4, 6, 4, 2, 1, 3, 1, 0, 8]

Accuracy: 0.08834178131788559
